# Chapter 6: Regression

**Foundations of Data Science**

## In this lesson

1. What a regression coefficient is: an adjusted comparison, in one number
2. One predictor: the line, the slope, and what R² does and doesn't tell you
3. Several predictors: holding things fixed
4. Categories and interactions
5. Diagnostics: is the model lying to you?
6. Collinearity: when two predictors say the same thing
7. Prediction vs explanation: train/validation/test and regularization

## The one idea

**A regression coefficient is the chapter 4 table, automated.** In chapter 4 we compared diabetes prevalence across income groups *within* age bands to remove the effect of age. A regression does that for every predictor at once: each coefficient is the change in the outcome per unit of that predictor, with all the other predictors held fixed. Everything else today is about when that number can be trusted.

New question for the clinic. Blood pressure is the other thing a screening visit measures, and the question is: **which patient characteristics predict systolic blood pressure, by how much, and how well can we predict an individual's reading?** The "by how much" is explanation; the "how well" is prediction. They use the same model and need different tools.

## Setup

<!-- nb-links -->
[Download this notebook](chapter_06_regression.ipynb?download=1){download="chapter_06_regression.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/chapter_06/chapter_06_regression.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Load and define the problem

Outcome: systolic blood pressure (`sbp`, mmHg), first reading. Predictors: age, BMI, waist, income ratio, diabetes status, sex, race/ethnicity. Adults with a blood pressure reading.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[(nhanes["age"] >= 20) & nhanes["sbp"].notna()].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["race_eth"] = adults["race_eth"].map({1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other/multi"})

print("adults with a BP reading:", len(adults))
adults[["sbp", "age", "bmi", "waist_cm", "income_ratio", "diabetes"]].describe().round(1).T

## 2. One predictor

Start with age, the strongest correlate from chapter 4. A straight line through the cloud:

$$\text{sbp} = \beta_0 + \beta_1 \cdot \text{age} + \varepsilon$$

Three numbers to read off: the **intercept** $\beta_0$ (predicted SBP at age 0, which is meaningless here, so we'll center age later), the **slope** $\beta_1$ (mmHg per year of age), and $\varepsilon$, the residual, which is everything about a person's blood pressure that age doesn't explain.

Fitting means choosing $\beta_0, \beta_1$ to make the residuals as small as possible, measured as the sum of squared residuals. That's "ordinary least squares." It has a closed-form answer (a formula, no search needed), which is why it's instant.

In [ ]:
from eda_toolkit import scatter_fit_plot

scatter_fit_plot(
    adults, x_vars=["age"], y_vars=["sbp"],
    add_best_fit_line=True, show_correlation=True,
    n_rows=1, n_cols=1, subplot_figsize=(7, 5), show_plot="subplots", **{"alpha": 0.25, "s": 10},
)

In [ ]:
# statsmodels formula interface: the formula reads like the equation. 'sbp ~ age' means sbp = b0 + b1*age.
ols_age = smf.ols("sbp ~ age", data=adults).fit()
print(ols_age.summary().tables[1])
print(f"\nR-squared: {ols_age.rsquared:.3f}    residual SD: {np.sqrt(ols_age.mse_resid):.1f} mmHg")

Reading the table:

- **coef** for `age` is about 0.5: each additional year of age is associated with about half a mmHg higher systolic pressure. Over 30 years, that's 15 mmHg, which is clinically real.
- **std err** and the **[0.025, 0.975]** columns are the standard error and 95% CI from chapter 3, applied to the slope. The interval is narrow because n is large.
- **P>|t|** tests whether the slope is zero. It's not. With n = 4,600 that was never in doubt.
- **R² = 0.22**: age explains 22% of the variation in SBP. The other 78% is everything else: genetics, salt, medication, the stress of being in a mobile exam center. A low R² does not mean the slope is wrong; it means age alone won't predict an individual well. Those are different questions.
- **Residual SD = 17 mmHg**: the typical miss when predicting a person's SBP from age alone. Compare to the SD of SBP itself (about 20) to see how little the line buys you for an individual.

## 3. Several predictors: holding things fixed

Add BMI, diabetes, sex. Each coefficient is now "per unit of this, with the others held constant." That is the adjustment we did by hand in chapter 4, done for all variables at once.

In [ ]:
ols_multi = smf.ols("sbp ~ age + bmi + diabetes + sex", data=adults).fit()
print(ols_multi.summary().tables[1])
print(f"\nR-squared: {ols_multi.rsquared:.3f}    residual SD: {np.sqrt(ols_multi.mse_resid):.1f} mmHg")

In [ ]:
# Crude vs adjusted: the diabetes coefficient alone, then with age in the model. Same trick as the chapter 4 heatmap.
crude = smf.ols("sbp ~ diabetes", data=adults).fit().params["diabetes"]
adjusted = ols_multi.params["diabetes"]
print(f"SBP difference, diabetes vs not:  crude {crude:.1f} mmHg   adjusted for age, BMI, sex {adjusted:.1f} mmHg")

The diabetes gap doesn't shrink; it disappears. 8 mmHg crude, -1 mmHg adjusted, with a confidence interval that comfortably includes zero. People with diabetes are older, and age was doing *all* of the work. That is confounding, measured exactly. (There's a second story hiding here too: many people with diabetes are on blood pressure medication, which lowers the reading. A variable we didn't include is pushing the coefficient down. Regression adjusts for what you give it, and only that.)

The `sex[T.male]` row is a category coefficient: men run about 2 mmHg higher than women at the same age, BMI, and diabetes status. statsmodels picked `female` as the reference level alphabetically; the coefficient is male relative to that.

What didn't change much: R² moved from 0.22 to 0.24, and the residual SD from 17.4 to 17.0. Three more predictors and the individual prediction is still bad. Keep that in mind for section 7.

## 4. Categories and interactions

Race/ethnicity has six levels, so it becomes five coefficients, each relative to a reference group. Chapter 4's heatmap suggested Black adults have higher BP at the same age; the model can say by how much, adjusted.

An **interaction** lets one predictor's slope depend on another. The chapter 4 findings log asked whether the sex gap changes with age. `age:sex` adds a term for that.

In [ ]:
# Center age so the intercept and the sex main effect are interpretable (at the mean age, ~50, not at age 0)
adults["age_c"] = adults["age"] - adults["age"].mean()

ols_full = smf.ols("sbp ~ age_c * sex + bmi + diabetes + C(race_eth, Treatment('White'))", data=adults).fit()
coefs = ols_full.summary2().tables[1][["Coef.", "Std.Err.", "[0.025", "0.975]", "P>|t|"]].round(3)
coefs.index = [i.replace("C(race_eth, Treatment('White'))", "race:") for i in coefs.index]
coefs

In [ ]:
# What the interaction means, drawn: predicted SBP by age for men and women, everything else at typical values
grid = pd.DataFrame({"age_c": np.tile(np.linspace(-30, 30, 61), 2), "sex": np.repeat(["female", "male"], 61)})
grid["bmi"] = adults["bmi"].median(); grid["diabetes"] = 0; grid["race_eth"] = "White"
grid["pred"] = ols_full.predict(grid)
grid["age"] = grid["age_c"] + adults["age"].mean()

fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=grid, x="age", y="pred", hue="sex", ax=ax)
ax.set_ylabel("Predicted SBP (mmHg)"); ax.set_title("Men start higher; women rise faster and overtake by 70 (the age x sex interaction)")
plt.show()

Read the rows together. `age_c` is 0.69: that's the slope for the reference sex, women, per year. `age_c:sex[T.male]` is -0.31, so men's slope is 0.69 - 0.31 = 0.38 per year. `sex[T.male]` is +2.1 *at the mean age* (that's why we centered). Put together: at 20, men are about 11 mmHg higher; the gap closes through the 50s; by 80 women are higher. That's a known pattern (menopause is the usual explanation), and the model found it from the interaction term alone. Without the term, the model would have reported one average slope and one average gap, both wrong for everyone.

Race coefficients: every group is higher than White adults at the same age, BMI, sex, and diabetes status. Black adults by 6.6 mmHg, the others by 2.5 to 3.6. These are *adjusted associations*, not causes; they're what's left after the listed predictors, and what's left includes everything we didn't measure.

## 5. Diagnostics: is the model lying?

A linear regression assumes the residuals are (a) centered on zero everywhere, not curving away from the line in some region; (b) similar in spread everywhere (**homoskedastic**); (c) roughly normal, which matters only for small samples; and (d) not dominated by a few influential rows. The library draws six panels that check all four, plus a summary table.

In [ ]:
from sklearn.linear_model import LinearRegression
from model_metrics import show_residual_diagnostics

# model_metrics works with sklearn estimators; refit the same model there so the diagnostic plots can use it.
# One-hot encode the categoricals by hand this once; the pipeline in section 7 will do it properly.
diag = adults.dropna(subset=["bmi"])
X_diag = pd.get_dummies(diag[["age", "bmi", "diabetes", "sex", "race_eth"]], drop_first=True, dtype=float)
y_diag = diag["sbp"]
lin = LinearRegression().fit(X_diag, y_diag)

show_residual_diagnostics(
    model=lin, X=X_diag, y=y_diag, model_title="OLS: sbp ~ age + bmi + diabetes + sex + race",
    plot_type="all", figsize=(14, 9), show_lowess=True,
    heteroskedasticity_test="breusch_pagan", show_diagnostics_table=True,
)

What the panels say here:

- **Residuals vs fitted** (top left): the smoothed line (lowess) is flat through most of the range and bends downward above fitted values of 140, meaning the model over-predicts for the people it expects to have the highest pressure. A mild sign that the relationship isn't perfectly linear at the top; a squared age term would be the thing to try.
- **Scale-location** (top right): the spread of residuals grows with the fitted value, and the Breusch-Pagan test confirms it (heteroskedastic). The coefficients are still unbiased; their standard errors are a bit off. Fix: `cov_type="HC3"` in statsmodels gives robust standard errors, shown below.
- **QQ plot** (top middle) and the **histogram** (bottom right): the right tail is heavier than normal, matching the skew we saw in chapter 3 (Jarque-Bera rejects normality). With n = 4,600 this doesn't affect inference on the coefficients.
- **Leverage and influence** (bottom left and middle): every point sits far inside the Cook's distance contours. No single person is steering the fit. The table's 325 "high leverage" rows are just the mechanical 2p/n threshold on a large sample, not a problem.

None of these is a reason to abandon the model. They are reasons to (a) use robust standard errors, (b) consider a curvature term, and (c) expect the model to miss high-BP people by more than it misses average ones, which matters if the clinic cares most about exactly those people.

In [ ]:
# Robust standard errors: same coefficients, honest intervals under unequal spread
ols_robust = smf.ols("sbp ~ age_c * sex + bmi + diabetes + C(race_eth, Treatment('White'))", data=adults).fit(cov_type="HC3")
compare = pd.DataFrame({"coef": ols_full.params, "se_classic": ols_full.bse, "se_robust": ols_robust.bse}).round(3)
compare.index = [i.replace("C(race_eth, Treatment('White'))", "race:") for i in compare.index]
compare

## 6. Collinearity

BMI and waist correlate at 0.91 (chapter 4). Put both in a model and each coefficient becomes "the effect of BMI holding waist fixed," which is a comparison the data barely contain: there are few people with high BMI and low waist. The coefficients get unstable, their standard errors balloon, and signs can flip. The **variance inflation factor** (VIF) measures it: how much a coefficient's variance is inflated by its correlation with the other predictors. Rule of thumb: VIF above 5 is a problem, above 10 is serious.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

both = smf.ols("sbp ~ age + bmi + waist_cm + diabetes + sex", data=adults.dropna(subset=["bmi", "waist_cm"])).fit()
print(both.summary().tables[1])

X_vif = both.model.exog
vif = pd.Series([variance_inflation_factor(X_vif, i) for i in range(X_vif.shape[1])], index=both.model.exog_names)
print("\nVIF:\n", vif.drop("Intercept").round(2))

BMI's coefficient barely moved, but its standard error nearly tripled (0.035 to 0.097), and waist came in at zero with a wide interval. The model can't tell the two apart, so it assigns the shared signal to one and reports uncertainty on both. VIFs of 7.4 and 7.7 confirm it. For *explanation*, keep one (the one the clinic actually measures). For *prediction*, it doesn't matter which, and regularization handles it automatically, which is where we go next.

## 7. Prediction vs explanation

Everything so far used all 4,600 people to fit and to judge. That's fine for explanation (we want the coefficients and their intervals). For prediction it's cheating: a model judged on the data it was fit to always looks better than it will on a new patient.

The fix is the **train / validation / test split**: fit on 60%, choose settings on 20%, report performance on the untouched 20% once. And **regularization**: ridge regression adds a penalty on the size of the coefficients, which shrinks them toward zero. Plainly: the model is told "don't trust any one predictor too much." A single number, alpha, controls how hard it shrinks; zero is ordinary least squares, large values flatten everything. The validation set picks alpha.

`model_tuner` wraps the split, the grid search over alpha, and the reporting. The preprocessing (impute, scale, one-hot) lives inside the pipeline so it's fit on the training rows only, which is the mistake students make most: imputing with the median of *all* the data leaks test information into training.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from model_tuner import Model

num = ["age", "bmi", "waist_cm", "income_ratio", "diabetes"]
cat = ["sex", "race_eth"]
X = adults[num + cat]
y = adults["sbp"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat),
])

ridge = Model(
    name="ridge_sbp", estimator_name="ridge", estimator=Ridge(), model_type="regression",
    pipeline_steps=[("pre", preprocess)],
    grid=[{"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]}],
    scoring=["neg_root_mean_squared_error"],
    train_size=0.6, validation_size=0.2, test_size=0.2, random_state=42, display=False,
)

ridge.grid_search_param_tuning(X, y)                 # picks alpha on the validation set
X_train, y_train = ridge.get_train_data(X, y)
X_test, y_test = ridge.get_test_data(X, y)
ridge.fit(X_train, y_train)
print("chosen alpha:", ridge.best_params_per_score["neg_root_mean_squared_error"]["params"])
print("rows: train", len(X_train), " test", len(X_test))

In [ ]:
from model_metrics import compute_regression_metrics

# Report on the test set, once. These are the numbers you'd quote.
y_pred = ridge.predict(X_test)
compute_regression_metrics(y_test, y_pred)

In [ ]:
# Honest vs dishonest: the same model scored on the rows it was trained on
y_pred_train = ridge.predict(X_train)
pd.DataFrame({
    "train (fit on these)": compute_regression_metrics(y_train, y_pred_train),
    "test (never seen)": compute_regression_metrics(y_test, y_pred),
}).loc[["RMSE", "MAE", "R^2"]]

Train and test RMSE are within 0.1 mmHg of each other; R² drops from 0.27 to 0.22. A linear model with eleven coefficients and 2,800 training rows can't overfit much. The gap is what to watch when the model is flexible (trees, boosting, next chapter) or the data are small (your project).

Bottom line for the clinic: the model predicts an individual's SBP to within about ±17 mmHg, which is not useful for an individual, and the clinic should keep measuring. What *is* useful is the adjusted coefficients: age, BMI, sex, and race/ethnicity shift expected pressure by amounts the clinic can plan around. Same model, two uses, two verdicts.

## What to carry forward

- A coefficient is an adjusted comparison; compare crude to adjusted to see confounding directly.
- R² is about prediction of individuals; coefficients and their CIs are about explanation. Don't let a low R² talk you out of a real effect, or a significant coefficient talk you into predicting individuals.
- Interactions let a slope depend on another variable; center continuous predictors first.
- Check residual plots; use robust SEs when spread is unequal.
- Collinear predictors: pick one for explanation, regularize for prediction.
- Fit preprocessing on training rows only; report test performance once.

## Looking ahead
- **Reading**: ISLR chapter 4 (classification), sections 4.1 through 4.4. Next chapter the outcome is diabetes yes/no, and the model is logistic regression.